# The textbook tables of a factorial: Yates's algorithm and three-level components

A textbook that analyses a factorial by hand prints its working as tables: the columns of Yates's
algorithm with the checks on them, and the components of an interaction of three-level factors.
The functions on this page return those tables, so each step can be set beside the book and
checked. For an analysis of variance with p-values, use `analyze_experiment`.

In [ ]:
import itertools

import matplotlib.pyplot as plt
import pandas as pd

from process_improve.experiments import interaction_components, reverse_yates, yates_table, yates_table_3k

## Yates's algorithm on a two-level factorial

Box, Hunter and Hunter (1978, Table 10.5) work the algorithm on a pilot plant: a $2^3$ factorial in
temperature T, concentration C and catalyst K, each response the average of two runs. Each pass
writes the sums of successive pairs of the column before, then their differences, the second of
each pair less the first. After three passes, column (3) holds the total and each effect's
contrast; the effect is its contrast divided by 4 (by 8 for the mean), and its sum of squares the
contrast squared divided by 8. The runs may come in any order: the table puts them in standard
order, T changing fastest.

In [ ]:
runs = pd.DataFrame(
    {
        "T": [-1, 1, -1, 1, -1, 1, -1, 1],
        "C": [-1, -1, 1, 1, -1, -1, 1, 1],
        "K": [-1, -1, -1, -1, 1, 1, 1, 1],
        "y": [60, 72, 54, 68, 52, 83, 45, 80],
    }
)
table = yates_table(runs, "y")
table

Each pass doubles the sum of squares of the column before it. The book uses this to check the
arithmetic:

In [ ]:
table.attrs["sum_of_squares"]

The table's column (3) and its effects are those of the book's Table 10.5:

In [ ]:
book_column_3 = [514, 92, -20, 6, 6, 40, 0, 2]
book_effects = [64.25, 23.0, -5.0, 1.5, 1.5, 10.0, 0.0, 0.5]
table["(3)"].tolist() == book_column_3, table["estimate"].tolist() == book_effects

## Fitted values by the algorithm run in reverse

Only T, C and the TK interaction stand out, and Box, Hunter and Hunter (Appendix 10A) fit the model
that keeps those three effects. With the other contrasts set to zero, the same passes, run on the
column written from the bottom up, give eight times the fitted values in standard order:

In [ ]:
fit = reverse_yates(table, keep=["T", "C", "T:K"])
fit[["T", "C", "K", "y", "effect", "kept", "fitted", "residual"]]

The crude sum of squares of the fitted values and the sum of squares of the residuals add up to
that of the responses:

In [ ]:
squares = fit.attrs["sum_of_squares"]
squares["fitted"], squares["residual"], squares["fitted"] + squares["residual"]

## Yates's algorithm on a three-level factorial

For a $3^k$ factorial each pass takes successive triples $(w_1, w_2, w_3)$ and writes their sums,
their linear contrasts $w_3 - w_1$, then their quadratic contrasts $w_1 - 2w_2 + w_3$. After $k$
passes each entry is $z'y$ for one product of the factors' orthogonal polynomials, the linear $x$
and the quadratic $3x^2 - 2$, so each of the $3^k - 1$ effects has one degree of freedom. The
effect's `estimate` is $z'y / z'z$, its coefficient in the least-squares fit, and `divisor` is
$z'z$.

Box and Draper (2007, Appendix 7G) work it on a textile experiment: the cycles to failure Y of
worsted yarn in a $3^3$ factorial in the length of the test specimen x1, the amplitude of the load
cycle x2 and the load x3. In the effect names, `x1^2` stands for the quadratic polynomial of x1.

In [ ]:
cycles = [674, 1414, 3636, 338, 1022, 1568, 170, 442, 1140, 370, 1198, 3184, 266, 620]
cycles += [1070, 118, 332, 884, 292, 634, 2000, 210, 438, 566, 90, 220, 360]
settings = [(x1, x2, x3) for x3, x2, x1 in itertools.product((-1, 0, 1), repeat=3)]
textile = pd.DataFrame(settings, columns=["x1", "x2", "x3"]).assign(Y=cycles)
table_3k = yates_table_3k(textile, "Y")
table_3k.round(2)

Box and Draper number the response column (1), so their columns (2) to (4) of Table 7G.1 are the
columns (1) to (3) here, and their columns (6), (7) and (8) are `divisor`, `estimate` and `sum_sq`.

`standardized` is $z'y / (z'z)^{1/2}$, the signed square root of the sum of squares, and
`normal_score` is the standard normal quantile at $(i - 1/2)/26$ of its rank $i$. Plotted against
each other they make the normal plot of the effects (Box and Draper, Figure 7F.1): the effects that
lie off the line through the small ones are those to look at. The largest, labelled, are the main
effects of the three factors and the interaction of length and amplitude.

In [ ]:
effects = table_3k.iloc[1:]
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(effects["standardized"], effects["normal_score"], s=20, color="tab:blue")
for _, row in effects[effects["standardized"].abs() > 1000].iterrows():
    left = row["standardized"] > 0  # the label on the side away from the edge of the plot
    ax.annotate(
        row["effect"],
        (row["standardized"], row["normal_score"]),
        xytext=(-6 if left else 6, -3),
        textcoords="offset points",
        ha="right" if left else "left",
        fontsize=8,
    )
ax.set_xlabel("Standardized effect, $z'y / (z'z)^{1/2}$")
ax.set_ylabel("Normal score")
plt.show()

## The components of an interaction of three-level factors

In a $3^k$ factorial the interaction of two factors has four degrees of freedom. Write each factor's
levels 0, 1 and 2, and group the runs by $x_A + x_B$ and by $x_A + 2x_B$, modulo 3: the sums of
squares of the three group totals are the interaction's two orthogonal components, AB and $AB^2$,
of two degrees of freedom each (Yates's J and I). An interaction of three factors splits into four
components, ABC, $ABC^2$, $AB^2C$ and $AB^2C^2$ (Yates's Z, Y, X and W).

Montgomery (2013, section 9.2) splits the interaction of the tool angle A and the cutting speed B
in his tool-life experiment (Example 5.5), a $3^2$ factorial run twice. A factor's levels are its
three settings, lowest first:

In [ ]:
tool_life = pd.DataFrame(
    {
        "A": [15] * 6 + [20] * 6 + [25] * 6,
        "B": [125, 125, 150, 150, 175, 175] * 3,
        "y": [-2, -1, -3, 0, 2, 3, 0, 2, 1, 3, 4, 6, -1, 0, 5, 6, 0, -1],
    }
)
components = interaction_components(tool_life, "y", ["A", "B"])
components.round(2)

Montgomery prints 28.00 for $AB^2$ and 33.34 for AB, where the table gives 33.33: his group totals
give $392/6 - 24^2/18 = 33.333$. The two components add up to the sum of squares of the
interaction, on its four degrees of freedom:

In [ ]:
interaction = components[components["effect"] == "AB"]
int(interaction["df"].sum()), round(float(interaction["sum_of_squares"].sum()), 2)

## The same tables as agent tools

Both tables are tools an agent can call through the MCP server or `skills/doe-designer/`:
`yates_table` takes a $2^k$ or a $3^k$ factorial, and with `keep` runs a $2^k$ in reverse;
`interaction_components` takes the runs and the three-level factors. Each takes the runs as a list
of records and returns the tables as records:

In [ ]:
from process_improve.tool_spec import execute_tool_call

request = {"design_matrix": runs.to_dict("records"), "response_column": "y", "keep": ["T", "C", "T:K"]}
result = execute_tool_call("yates_table", request)
[row["fitted"] for row in result["reverse"]["table"]]